In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

In [ ]:
df = pd.read_csv("solar_system.csv")
print("Shape before:", df.shape)

In [ ]:
df = df.set_index("Attribute").T
df.index.name = "Planet"
df.reset_index(inplace=True)
df.columns.name = None

In [ ]:
print("Shape after:", df.shape)

In [ ]:
# 5
# a. Before: something like (19, 11) — 19 rows (attributes) × 11 columns (planets + the label column). 
#After: something like (10, 20) — 10 rows (planets) × 20 columns (attributes + Planet column)

# b. Before: 19 rows, 11 columns. After: 10 rows, 20 columns

# c. 
# df.set_index("Attribute").T — Sets the "Attribute" column as the index (so attribute names like "Mass", "Diameter" etc. become row labels), then .T transposes the entire DataFrame, flipping rows and columns so planets become rows and attributes become columns
# df.index.name = "Planet" — After transposing, the index (which is now planet names) has no label, so this renames it to "Planet"
# df.reset_index(inplace=True) — Moves "Planet" out of the index and into a regular column so it's accessible like any other column
# df.columns.name = None — After all the transposing, pandas leaves a leftover name on the columns axis (it keeps the old index name "Attribute"). This line clears that so the columns axis has no label

# d. The shape is (10, 20) instead of (11, 20) because when you call set_index("Attribute"), the "Attribute" column is removed
# from the data and becomes the index. So that column no longer counts as a data column, it becomes the axis labels. You lose one from
# the count, giving you 10 planets instead of 11 rows.

# e. 
print(df.columns.tolist())

In [ ]:
# 5.
# f. Columns with units: Mass (10^24kg), Diameter (km), Density (kg/m3), Gravity (m/s2), Escape Velocity (km/s), Rotation Period (hours), 
# Length of Day (hours), Distance from Sun (10^6 km), Perihelion (10^6 km), Aphelion (10^6 km), Orbital Period (days), Orbital Velocity (km/s),
# Orbital Inclination (degrees), Orbital Eccentricity (dimensionless but labeled), Obliquity to Orbit (degrees), Mean Temperature (C), 
# Surface Pressure (bars)

# Columns without units: Planet (name, text), Number of Moons (a count, no physical unit), Ring System? (Yes/No), Global Magnetic Field? (Yes/No)

# g.
print(df.to_string())

In [ ]:
# 2.1.2
print(df.dtypes)

In [ ]:
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].apply(type))

In [ ]:
# a. Even though the values look like numbers (e.g., 3.3, 6371), they are all stored as <class 'str'> (strings). The "Planet", "Ring System?", and "Global Magnetic Field?" columns are also strings but that's expected.

# b. Storing numeric values as strings a problem because you cannot perform any mathematical operations on strings.

In [ ]:
# 2.1.3
for col in df.columns:
    if col not in ["Planet", "Ring System?", "Global Magnetic Field?"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
print(df.dtypes)

In [ ]:
# a. After conversion, you'll see: float64, int64, and object

# b. Define each one:
# float64: A 64-bit floating point number, meaning it can store decimal values (e.g., 3.301, 4.867). Most planetary measurements will be this type since they have decimal places
# int64: A 64-bit integer, meaning whole numbers only (e.g., 0, 1, 27). Columns like "Number of Moons" will likely be this type
# object: A generic Python object type. In pandas, this almost always means the column contains strings (text). Used for columns with non-numeric data

# c. 
# "Planet" — Still object because it contains planet names (text like "Mercury", "Venus", etc.), which can't be converted to a number
# "Ring System?" — Still object because it contains Yes/No text values, not numbers
# "Global Magnetic Field?" — Still object for the same reason — Yes/No answers are text

In [ ]:
# 2.2
def attach_units(df, old_col, new_col, unit):
    new_values = []
    for val in df[old_col]:
        new_values.append(val * unit)
    df[old_col] = new_values
    df.rename(columns={old_col: new_col}, inplace=True)

attach_units(df, "Mass (10^24kg)", "Mass (kg)", 1e24 * u.kg)
print(df["Mass (kg)"])

In [ ]:
# 2.3
# Compute average of perihelion and aphelion
semi_major = (df["Perihelion (10^6 km)"] + df["Aphelion (10^6 km)"]) / 2

# Insert right after Aphelion column
aph_idx = df.columns.get_loc("Aphelion (10^6 km)")
df.insert(aph_idx + 1, "Semi-Major Axis (10^6 km)", semi_major)

# Confirm insertion
print(df.columns.tolist())

# Print the new column
print(df["Semi-Major Axis (10^6 km)"])

In [ ]:
# Convert days to years using Astropy
df["Orbital Period (days)"] = df["Orbital Period (days)"].apply(
    lambda x: (x * u.day).to(u.yr).value
)

# Rename the column
df.rename(columns={"Orbital Period (days)": "Orbital Period (years)"}, inplace=True)

# Print all column names to confirm rename
print(df.columns.tolist())

# Print the full column
print(df["Orbital Period (years)"])

# Print favorite planet's orbital period to 4 decimal places (using Earth)
earth = df[df["Planet"] == "Earth"].iloc[0]
print(f"Earth's orbital period: {earth['Orbital Period (years)']:.4f} years")

In [ ]:
# 2.2.3
# AU: An Astronomical Unit (AU) is the average distance between the Earth and the Sun

print(const.au)

In [ ]:
print(const.au.to(u.km))

In [ ]:
# Columns in 10^6 km
large_dist_cols = {
    "Perihelion (10^6 km)": "Perihelion (AU)",
    "Aphelion (10^6 km)": "Aphelion (AU)",
    "Semi-Major Axis (10^6 km)": "Semi-Major Axis (AU)"
}

for old_col, new_col in large_dist_cols.items():
    df[old_col] = df[old_col].apply(
        lambda x: (x * 1e6 * u.km).to(u.au).value
    )
    df.rename(columns={old_col: new_col}, inplace=True)

# Diameter is just in km (not 10^6 km)
df["Diameter (km)"] = df["Diameter (km)"].apply(
    lambda x: (x * u.km).to(u.au).value
)
df.rename(columns={"Diameter (km)": "Diameter (AU)"}, inplace=True)

# Print a sample value for your favorite planet (Earth)
earth = df[df["Planet"] == "Earth"].iloc[0]
for col in ["Perihelion (AU)", "Aphelion (AU)", "Semi-Major Axis (AU)", "Diameter (AU)"]:
    print(f"Earth {col}: {earth[col]:.6f} AU")

In [ ]:
# Bonus
# Surface Gravity = GM/R^2
G = const.G

mass_kg  = df["Mass (kg)"].apply(lambda x: x.value if hasattr(x, 'value') else x)
radius_m = df["Diameter (AU)"].apply(
    lambda x: ((x * u.au).to(u.m) / 2).value
)
mass_g = mass_kg * 1000  # convert kg to grams for x-axis

gravity = []
for m, r in zip(mass_kg, radius_m):
    g = (const.G.value * m) / (r ** 2)
    gravity.append(g)

plt.figure(figsize=(8, 6))
plt.loglog(mass_g, gravity, 'o', markersize=8)

# Label each planet
for i, row in df.iterrows():
    plt.annotate(row["Planet"], (mass_g.iloc[i], gravity[i]), fontsize=8)

plt.xlabel("Mass (g)")
plt.ylabel("Surface Gravity (m/s²)")
plt.title("Surface Gravity vs Mass")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Bonus Bonus!!
from scipy.optimize import curve_fit

log_mass    = np.log10(mass_g)
log_gravity = np.log10(gravity)

# Linear model for log-log fit
def linear(x, m, b):
    return m * x + b

params, _ = curve_fit(linear, log_mass, log_gravity)
print(f"Slope: {params[0]:.4f}")
print(f"Intercept: {params[1]:.4f}")

# Reduced chi-squared
predicted   = linear(log_mass, *params)
residuals   = log_gravity - predicted
chi2        = np.sum(residuals**2)
reduced_chi2 = chi2 / (len(log_gravity) - len(params))
print(f"Reduced chi-squared: {reduced_chi2:.4f}")

# Plot data + fit
plt.figure(figsize=(8, 6))
plt.loglog(mass_g, gravity, 'o', label="Data", markersize=8)
plt.loglog(mass_g, 10**linear(log_mass, *params), '-', label="Best Fit")
plt.xlabel("Mass (g)")
plt.ylabel("Surface Gravity (m/s²)")
plt.title("Surface Gravity vs Mass with Best Fit")
plt.legend()
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# 2.4
df.to_csv("units.csv", index=False)